### get poster and boxoffice data from tmdb api

In [ ]:
import requests
import pandas as pd

TMDB_API_KEY = "your_key"

def fetch_tmdb_info(df, cols, tmdb_col="tmdbId"):
    output = df.copy()

    def fetch_one(tmdb_id):
        if pd.isna(tmdb_id):
            return pd.Series({c: None for c in cols})

        try:
            r = requests.get(
                f"https://api.themoviedb.org/3/movie/{int(tmdb_id)}",
                params={"api_key": TMDB_API_KEY},
                timeout=15
            )

            if r.status_code != 200:
                return pd.Series({c: None for c in cols})

            data = r.json()

            result = {}
            for c in cols:
                if c in ["poster_path", "backdrop_path"]:
                    path = data.get(c)
                    result[c] = f"https://image.tmdb.org/t/p/w500{path}" if path else None
                else:
                    result[c] = data.get(c)

            return pd.Series(result)

        except:
            return pd.Series({c: None for c in cols})

    output[cols] = output[tmdb_col].apply(fetch_one)
    return output


movie_cols = [
  "title",
  "original_title",
  "release_date",
  "budget",
  "revenue",
  "poster_path",
  "backdrop_path",
  "imdb_id"     # this is the imdbId return back from tmdb api
]

poster_col = [
    "poster_path"
]

boxoffic_col = ["budget", "revenue"]


In [ ]:
# load original links.csv
links = pd.read_csv("links.csv", dtype=str)

In [ ]:
# get poster + boxoffice
df = fetch_tmdb_info(links, poster_col + boxoffic_col)
df

In [ ]:
# save poster.csv
poster = df[["imdbId", "poster_path"]]
poster.to_csv("poster.csv", index=False)

# save boxoffice.csv
boxoffice = df[["imdbId", "budget", "revenue"]]
boxoffice.to_csv("boxoffice.csv", index=False)

### get awards and nominations from wikidata api

In [ ]:
import pandas as pd
import requests

def get_wikidata_awards_and_noms(imdb_id):
    imdb_id = str(imdb_id).strip()
    if not imdb_id.startswith("tt"):
        imdb_id = "tt" + imdb_id

    query = f"""
    SELECT DISTINCT ?type ?itemLabel WHERE {{
      ?film wdt:P345 "{imdb_id}" .
      {{
        ?film wdt:P166 ?item .
        BIND("award_received" AS ?type)
      }}
      UNION
      {{
        ?film wdt:P1411 ?item .
        BIND("nominated_for" AS ?type)
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "UCL-DBMS-Coursework/1.0 (student project)"
    }
    r = requests.get(url, params={"query": query}, headers=headers, timeout=30)
    r.raise_for_status()

    rows = [(b["type"]["value"], b["itemLabel"]["value"]) for b in r.json()["results"]["bindings"]]
    return rows if rows else None

# only for testing, this movie is godfather
get_wikidata_awards_and_noms("0068646")

[('award_received', 'Golden Globe Award for Best Motion Picture – Drama'),
 ('award_received', 'Academy Award for Best Picture'),
 ('nominated_for', 'Academy Award for Best Picture'),
 ('nominated_for', 'Academy Award for Best Director'),
 ('award_received', 'Academy Award for Best Actor'),
 ('nominated_for', 'Academy Award for Best Actor'),
 ('nominated_for', 'Academy Award for Best Supporting Actor'),
 ('award_received', 'Academy Award for Best Writing, Adapted Screenplay'),
 ('nominated_for', 'Academy Award for Best Writing, Adapted Screenplay'),
 ('award_received', 'National Board of Review: Top Ten Films'),
 ('nominated_for', 'Academy Award for Best Costume Design'),
 ('nominated_for', 'Academy Award for Best Film Editing'),
 ('award_received',
  'Golden Globe Award for Best Actor – Motion Picture Drama'),
 ('nominated_for', 'Academy Award for Best Sound'),
 ('award_received', 'Golden Globe Award for Best Screenplay')]

In [ ]:
# load the origin links.csv
links = pd.read_csv("links.csv", dtype=str)

In [ ]:
# get and save the first 1000 rows to csv
df = links.iloc[0:1000]
df["awards_noms"] = df["imdbId"].apply(get_wikidata_awards_and_noms)
awards = df[["imdbId", "awards_noms"]]
awards.to_csv("awards_json.csv", index=False)

In [ ]:
# rerun this code block to get the rest data, change the links.iloc[x:y] to manually repeat, e.g. links.iloc[1000:2000], links.iloc[2000:3000]
# if runtime error, u can shorten the interval in links.iloc, e.g. links.iloc[1000:1200], links.iloc[1200:1400]
df = links.iloc[1000:2000]
df["awards_noms"] = df["imdbId"].apply(get_wikidata_awards_and_noms)
awards = df[["imdbId", "awards_noms"]]
awards.to_csv("awards_json.csv", mode="a", index=False, header=False)

In [ ]:
# turn this to a sql base csv, we will use this file eventually
import pandas as pd
import ast

df = pd.read_csv("awards_json.csv", dtype=str)

rows = []

for _, row in df.iterrows():
    if pd.isna(row["awards_noms"]) or str(row["awards_noms"]).strip() == "":
        continue

    try:
        awards_list = ast.literal_eval(row["awards_noms"])
    except Exception:
        continue

    for award_type, award_name in awards_list:
        rows.append((row["imdbId"], award_type, award_name))

new_df = pd.DataFrame(rows, columns=["imdbId", "award_type", "award_name"])
new_df = new_df.drop_duplicates()

new_df.to_csv("awards_noms.csv", index=False)

### Validation

In [3]:
import pandas as pd
links = pd.read_csv("links.csv", dtype=str)
poster = pd.read_csv("poster.csv", dtype=str)
boxoffice = pd.read_csv("boxoffice.csv", dtype=str)
awards = pd.read_csv("awards_json.csv", dtype=str)

# validate all csv data and original links has the exact same imdbId rows
def validate(df1: pd.DataFrame, df2: pd.DataFrame, col: str = 'imdbId'):
    df1_m = set(df1[col]) - set(df2[col])
    df2_m= set(df2[col]) - set(df1[col])
    print("Missing in df1 in df2:", df1_m)
    print("Missing in df2 in df1:", df2_m)

# should be all empty
validate(links, poster, "imdbId")
validate(links, boxoffice, "imdbId")
validate(links, awards, "imdbId")

Missing in df1 in df2: set()
Missing in df2 in df1: set()
Missing in df1 in df2: set()
Missing in df2 in df1: set()
Missing in df1 in df2: set()
Missing in df2 in df1: set()
